In [1]:
import os

In [3]:
%pwd

'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\research'

In [4]:
os.chdir("../")

In [12]:
%pwd

'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    checkpoint_model_filepath: Path
    tensorboard_root_log_dir: Path

In [17]:
import sys
print(sys.path)

['C:\\Users\\MATT\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\MATT\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\MATT\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'C:\\Users\\MATT\\AppData\\Local\\Programs\\Python\\Python312', 'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\venv', '', 'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\venv\\Lib\\site-packages', 'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\venv\\Lib\\site-packages\\win32', 'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\venv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification\\venv\\Lib\\site-packages\\Pythonwin']


In [ ]:
    import sys
    import os

    # Add src directory to Python path
    src_path = os.path.join(os.getcwd(), 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

print("Added src to Python path:", src_path)

# Now your imports should work
from Chicken_Disease_Classification.utils.common import read_yaml, create_directories
from Chicken_Disease_Classification.constant import *

print("✅ Imports successful!")

Added src to Python path: c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\src
✅ Imports successful!


In [23]:
from Chicken_Disease_Classification.utils.common import read_yaml, create_directories 
from Chicken_Disease_Classification.constant import *


In [34]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([Path(str(self.config.artifacts_root))])

    
    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        prepare_callbacks_config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(prepare_callbacks_config.checkpoint_model_filepath)
        create_directories([
            
            Path(str(prepare_callbacks_config.root_dir)),
            Path(str(model_ckpt_dir)), 
            Path(str(prepare_callbacks_config.tensorboard_root_log_dir))
        ])
        
        
        
        prepare_callbacks_config = PrepareCallbacksConfig(
            root_dir=Path(prepare_callbacks_config.root_dir),
            checkpoint_model_filepath=Path(prepare_callbacks_config.checkpoint_model_filepath),
            tensorboard_root_log_dir=Path(prepare_callbacks_config.tensorboard_root_log_dir)
        )
        
        return prepare_callbacks_config

In [31]:

import os
import urllib.request as request 
from Chicken_Disease_Classification.utils.logger import logger
from Chicken_Disease_Classification.utils.common import get_size
import tensorflow as tf


In [35]:
import os
import time
import tensorflow as tf
from pathlib import Path


class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config

    @property
    def create_tb_callbacks(self):
        """Create TensorBoard callback with timestamped log directory"""
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}"
        )
        
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)

    @property
    def create_ckpt_callbacks(self):
        """Create ModelCheckpoint callback"""
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=self.config.checkpoint_model_filepath,
            save_best_only=True
        )

    def get_tb_ckpt_callbacks(self):
        """Get both TensorBoard and ModelCheckpoint callbacks"""
        return [
            self.create_tb_callbacks,
            self.create_ckpt_callbacks
        ]

In [36]:
try:
    config_manager = ConfigurationManager()
    prepare_callbacks_config = config_manager.get_prepare_callbacks_config()
    prepare_callbacks =    PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
except Exception as e:
    raise e

[2025-09-14 19:14:06,817: INFO: common: Directory created at: artifacts]
[2025-09-14 19:14:06,820: INFO: common: Directory created at: artifacts\prepare_callbacks]
[2025-09-14 19:14:06,824: INFO: common: Directory created at: artifacts\prepare_callbacks]
[2025-09-14 19:14:06,827: INFO: common: Directory created at: artifacts\prepare_callbacks\tensorboard_logs_dirs]
